# Level 2 Task 1: Regression Analysis

This notebook performs a simple linear regression analysis to predict house prices using the housing dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data" / "house_prediction_dataset.csv").exists():
    base_dir = Path("Level_2_Intermediate") / "Task_1_Regression_Analysis"

data_path = base_dir / "data" / "house_prediction_dataset.csv"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

column_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]

In [ ]:
df = pd.read_csv(data_path, sep=r"\s+", header=None, names=column_names)
print("Dataset shape:", df.shape)
df.head()

## Select Predictor and Target

In [ ]:
feature_name = "RM"
target_name = "MEDV"

x = df[feature_name].to_numpy()
y = df[target_name].to_numpy()

df[[feature_name, target_name]].corr().round(3)

## Train-Test Split

In [ ]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(df))
split_index = int(len(df) * 0.8)
train_idx, test_idx = indices[:split_index], indices[split_index:]
x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print("Training rows:", len(x_train))
print("Testing rows:", len(x_test))

## Fit Simple Linear Regression

In [ ]:
design_matrix = np.column_stack([np.ones(len(x_train)), x_train])
intercept, slope = np.linalg.lstsq(design_matrix, y_train, rcond=None)[0]
y_pred = intercept + slope * x_test
print(f"Intercept: {intercept:.4f}")
print(f"Slope: {slope:.4f}")

## Evaluate the Model

In [ ]:
mse = np.mean((y_test - y_pred) ** 2)
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

## Regression Plot

In [ ]:
sorted_indices = np.argsort(x_test)
x_sorted = x_test[sorted_indices]
y_pred_sorted = y_pred[sorted_indices]

plt.figure(figsize=(9, 6))
plt.scatter(x_train, y_train, alpha=0.7, color="#3498db", label="Training data")
plt.scatter(x_test, y_test, alpha=0.8, color="#2ecc71", label="Testing data")
plt.plot(x_sorted, y_pred_sorted, color="#e74c3c", linewidth=2.5, label="Regression line")
plt.title("Simple Linear Regression: RM vs MEDV", fontweight="bold")
plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("Median Home Value (MEDV)")
plt.legend()
plt.tight_layout()
plt.show()

## Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, color="#8e44ad", alpha=0.8)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="black", linewidth=1.5)
plt.title("Actual vs Predicted Home Prices", fontweight="bold")
plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")
plt.tight_layout()
plt.show()